# Solar Module Dataset EDA

This notebook explores the local `InfraredSolarModules` dataset stored in `classwork/solar/dataset`. It keeps the analysis aligned with the existing training notebook while staying focused on dataset understanding: schema, class balance, image integrity, image dimensions, and representative samples.

Two label views are built from the same metadata:
- `anomaly_class`: original 12-class labels from `module_metadata.json`
- `status`: binary label derived from `anomaly_class` where `No-Anomaly -> Normal` and everything else -> `Faulty`

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["image.cmap"] = "inferno"


def find_solar_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        local_dataset = candidate / "dataset" / "2020-02-14_InfraredSolarModules" / "InfraredSolarModules"
        repo_dataset = candidate / "classwork" / "solar" / "dataset" / "2020-02-14_InfraredSolarModules" / "InfraredSolarModules"
        if local_dataset.exists():
            return candidate
        if repo_dataset.exists():
            return candidate / "classwork" / "solar"
    raise FileNotFoundError("Could not locate classwork/solar relative to the current working directory.")


SOLAR_DIR = find_solar_dir(Path.cwd())
DATASET_DIR = SOLAR_DIR / "dataset" / "2020-02-14_InfraredSolarModules" / "InfraredSolarModules"
METADATA_PATH = DATASET_DIR / "module_metadata.json"
IMAGES_DIR = DATASET_DIR / "images"

print(f"Solar directory: {SOLAR_DIR}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Metadata file exists: {METADATA_PATH.exists()}")
print(f"Images directory exists: {IMAGES_DIR.exists()}")

In [ ]:
with METADATA_PATH.open() as handle:
    raw_metadata = json.load(handle)

df = (
    pd.DataFrame.from_dict(raw_metadata, orient="index")
    .rename_axis("module_id")
    .reset_index()
)

df["module_id"] = df["module_id"].astype(int)
df["image_path"] = df["image_filepath"].map(lambda rel_path: DATASET_DIR / rel_path)
df["status"] = np.where(df["anomaly_class"].eq("No-Anomaly"), "Normal", "Faulty")
df["file_exists"] = df["image_path"].map(Path.exists)

dataset_summary = pd.Series(
    {
        "rows": len(df),
        "columns": df.shape[1],
        "unique anomaly classes": df["anomaly_class"].nunique(),
        "unique binary status labels": df["status"].nunique(),
        "missing image files": (~df["file_exists"]).sum(),
    }
)

display(dataset_summary.to_frame(name="value"))
display(df.head())

In [ ]:
class_counts = (
    df["anomaly_class"]
    .value_counts()
    .rename_axis("anomaly_class")
    .reset_index(name="count")
)
class_counts["pct"] = class_counts["count"] / len(df) * 100

status_counts = (
    df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)
status_counts["pct"] = status_counts["count"] / len(df) * 100

_, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=class_counts, y="anomaly_class", x="count", ax=axes[0], color="#d95f02")
axes[0].set_title("Original anomaly_class distribution")
axes[0].set_xlabel("Number of images")
axes[0].set_ylabel("Class")

sns.barplot(data=status_counts, x="status", y="count", ax=axes[1], hue="status", dodge=False, palette=["#1b9e77", "#7570b3"], legend=False)
axes[1].set_title("Derived binary status distribution")
axes[1].set_xlabel("Status")
axes[1].set_ylabel("Number of images")
for patch in axes[1].patches:
    axes[1].annotate(
        f"{int(patch.get_height()):,}",
        (patch.get_x() + patch.get_width() / 2, patch.get_height()),
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

print(
    "Multiclass imbalance ratio (largest / smallest): "
    f"{class_counts['count'].max() / class_counts['count'].min():.2f}"
)
display(class_counts)
display(status_counts)

In [ ]:
def image_metadata(path: Path) -> dict:
    with Image.open(path) as img:
        image_mode = img.mode
        width, height = img.size
        array = np.asarray(img)
    return {
        "mode": image_mode,
        "width": width,
        "height": height,
        "min_pixel": int(array.min()),
        "max_pixel": int(array.max()),
        "mean_pixel": float(array.mean()),
    }


image_audit = pd.DataFrame(df["image_path"].map(image_metadata).tolist())
eda_df = pd.concat([df, image_audit], axis=1)

audit_summary = {
    "unique modes": eda_df["mode"].value_counts().to_dict(),
    "unique widths": sorted(eda_df["width"].unique().tolist()),
    "unique heights": sorted(eda_df["height"].unique().tolist()),
    "mean pixel intensity": round(eda_df["mean_pixel"].mean(), 2),
    "global min pixel": int(eda_df["min_pixel"].min()),
    "global max pixel": int(eda_df["max_pixel"].max()),
}

display(pd.Series(audit_summary, name="value").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=eda_df, x="mean_pixel", hue="status", bins=30, kde=True, ax=axes[0])
axes[0].set_title("Mean image intensity by binary status")

sampled_for_boxplot = eda_df.sample(min(1500, len(eda_df)), random_state=42)
sns.boxplot(data=sampled_for_boxplot, x="status", y="mean_pixel", hue="status", dodge=False, ax=axes[1], palette=["#1b9e77", "#7570b3"], legend=False)
axes[1].set_title("Sampled mean intensity spread")

plt.tight_layout()
plt.show()

In [ ]:
def show_examples(frame: pd.DataFrame, group_col: str, ncols: int = 4, seed: int = 42) -> None:
    examples = (
        frame.groupby(group_col, group_keys=False)
        .sample(n=1, random_state=seed)
        .sort_values(group_col)
        .reset_index(drop=True)
    )

    nrows = int(np.ceil(len(examples) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, (_, row) in zip(axes, examples.iterrows()):
        with Image.open(row["image_path"]) as img:
            ax.imshow(np.asarray(img))
        ax.set_title(f"{row[group_col]}\nmodule_id={row['module_id']}")
        ax.axis("off")

    for ax in axes[len(examples):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_examples(eda_df, group_col="anomaly_class")

In [ ]:
takeaways = pd.DataFrame(
    [
        {
            "finding": "Dataset size",
            "value": f"{len(eda_df):,} images",
        },
        {
            "finding": "Original label space",
            "value": f"{eda_df['anomaly_class'].nunique()} classes",
        },
        {
            "finding": "Binary label space",
            "value": f"{eda_df['status'].nunique()} classes (Normal vs Faulty)",
        },
        {
            "finding": "Image dimensions",
            "value": f"{eda_df['height'].mode().iloc[0]} x {eda_df['width'].mode().iloc[0]} pixels",
        },
        {
            "finding": "Most common multiclass label",
            "value": class_counts.iloc[0]['anomaly_class'] + f" ({class_counts.iloc[0]['count']:,})",
        },
        {
            "finding": "Rarest multiclass label",
            "value": class_counts.iloc[-1]['anomaly_class'] + f" ({class_counts.iloc[-1]['count']:,})",
        },
    ]
)

display(takeaways)

## Training-Oriented Checks

The original CNN notebook uses a class-wise split over `anomaly_class`, so the EDA should verify what that split looks like in practice. The next cells reproduce that logic, summarize label balance across train/validation/test, and highlight what that means for model design and evaluation.

In [ ]:
def split_like_training_notebook(dataframe: pd.DataFrame):
    df_train = pd.DataFrame(columns=dataframe.columns)
    df_test = pd.DataFrame(columns=dataframe.columns)
    df_val = pd.DataFrame(columns=dataframe.columns)

    shuffled = dataframe.sample(frac=1, random_state=42).reset_index(drop=True)

    for anomaly_class, group in shuffled.groupby("anomaly_class"):
        total = len(group)
        test_size = int(total * 0.14)
        val_size = int(total * 0.01)

        group = group.sample(frac=1, random_state=42).reset_index(drop=True)
        test_split = group.iloc[:test_size]
        val_split = group.iloc[test_size:test_size + val_size]
        train_split = group.iloc[test_size + val_size:]

        df_train = pd.concat([df_train, train_split], ignore_index=True)
        df_test = pd.concat([df_test, test_split], ignore_index=True)
        df_val = pd.concat([df_val, val_split], ignore_index=True)

    return df_train, df_val, df_test


split_frames = dict(zip(["train", "validation", "test"], split_like_training_notebook(eda_df)))
split_summary = pd.DataFrame(
    {
        split_name: [
            len(split_frame),
            split_frame["status"].eq("Normal").sum(),
            split_frame["status"].eq("Faulty").sum(),
            split_frame["anomaly_class"].nunique(),
        ]
        for split_name, split_frame in split_frames.items()
    },
    index=["rows", "normal", "faulty", "unique anomaly classes"],
)

split_status_counts = pd.concat(
    [
        split_frame["status"].value_counts().rename(split_name)
        for split_name, split_frame in split_frames.items()
    ],
    axis=1,
).fillna(0).astype(int)

split_class_counts = pd.concat(
    [
        split_frame["anomaly_class"].value_counts().rename(split_name)
        for split_name, split_frame in split_frames.items()
    ],
    axis=1,
).fillna(0).astype(int)

train_ids = set(split_frames["train"]["module_id"])
val_ids = set(split_frames["validation"]["module_id"])
test_ids = set(split_frames["test"]["module_id"])

leakage_summary = pd.Series(
    {
        "train ∩ validation": len(train_ids & val_ids),
        "train ∩ test": len(train_ids & test_ids),
        "validation ∩ test": len(val_ids & test_ids),
    },
    name="overlap_count",
)

display(split_summary)
display(split_status_counts)
display(split_class_counts)
display(leakage_summary.to_frame())

In [ ]:
binary_examples = (
    eda_df.groupby("status", group_keys=False)
    .sample(n=6, random_state=42)
    .sort_values(["status", "anomaly_class", "module_id"])
    .reset_index(drop=True)
)

_, axes = plt.subplots(2, 6, figsize=(18, 6))
axes = axes.ravel()

for ax, (_, row) in zip(axes, binary_examples.iterrows()):
    with Image.open(row["image_path"]) as img:
        ax.imshow(np.asarray(img), cmap="inferno")
    ax.set_title(f"{row['status']}\n{row['anomaly_class']}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

modeling_implications = pd.DataFrame(
    [
        {
            "topic": "Binary balance",
            "implication": "Normal and Faulty are effectively balanced, so class weighting is optional for the binary task.",
        },
        {
            "topic": "Multiclass imbalance",
            "implication": "The 12-class view is highly imbalanced, so any multiclass model should track per-class metrics, not only accuracy.",
        },
        {
            "topic": "Image geometry",
            "implication": "Images have a fixed low resolution, so compact CNNs and careful downsampling are more appropriate than very deep image backbones.",
        },
        {
            "topic": "Validation size",
            "implication": "The training notebook's 1% validation split is small; metrics from that split will be noisy compared with a larger validation set.",
        },
        {
            "topic": "Leakage check",
            "implication": "The reproduced split has zero module_id overlap across train, validation, and test, so the split logic is data-leakage safe at the image level.",
        },
    ]
)

display(modeling_implications)